In [1]:
import re
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords


STEMMER = SnowballStemmer(
    language = "english",
    #ignore_stopwords = False
)

TOKENIZER = TweetTokenizer(
    preserve_case=False,  
    reduce_len=True,      
    strip_handles=True,  
    # match_phone_numbers = false;
)

URL_RE = re.compile(r"^(https?://|www\.)")
HAS_AZ_RE = re.compile(r"[a-z]")


def stem_tokenize(text):
    tokens = TOKENIZER.tokenize(text)

    stems = []
    for tok in tokens:
        #remove urls
        if URL_RE.match(tok):
            continue

        # This removes the #, I am not sure if we want to remove so the words in # get tokenized like normal,
        # or if we want to some way signify that there is a # in there
        tok = tok.lstrip("#")

        # must have char a-z
        if not HAS_AZ_RE.search(tok):
            continue

        stems.append(STEMMER.stem(tok))

    return stems

# load data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# fill missing data
X_train = train_df["text"].fillna("")
y_train = train_df["target"]
X_test = test_df["text"].fillna("")

base_stop_words = set(stopwords.words("english"))

keep_words = {
    'not', 'no', 'nor',
    'up', 'down', 'out', 'off',
    'after', 'before', 'during',
    'again', 'more', 'all',
    'over', 'only'
}
base_stop_words -= keep_words

STOP_WORDS = sorted({stop_w for words in base_stop_words for stop_w in stem_tokenize(words)})

vectorizer = CountVectorizer(
    tokenizer=stem_tokenize,
    token_pattern=None,      
    lowercase=False,    
    stop_words=STOP_WORDS
)
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

Prints to check

In [2]:
# Print an example to ensure it is working
print("Example tokenization:")
for t in X_train.sample(3, random_state=0):
    print(f"Raw tweet: {t}")
    print(f"Tokenized and stemmed: {stem_tokenize(t)}\n")
    

print("Train matrix shape:", X_train_counts.shape)
print("Test matrix shape: ", X_test_counts.shape)
print("First 20 stems in vocabulary:", vectorizer.get_feature_names_out()[:20])

print("X Train Sparse Matrix Shape:", X_train_counts.shape)
print("X Train Dense Matrix Shape:", X_train_counts.todense().shape)
print("Y Train Shape:", y_train.shape)

Example tokenization:
Raw tweet: @KatieKatCubs you already know how this shit goes. World Series or Armageddon.
Tokenized and stemmed: ['you', 'alreadi', 'know', 'how', 'this', 'shit', 'goe', 'world', 'seri', 'or', 'armageddon']

Raw tweet: @LeMaireLee @danharmon People Near Meltdown Comics Who Have Free Time to Wait in Line on Sunday Nights are not a representative sample. #140
Tokenized and stemmed: ['peopl', 'near', 'meltdown', 'comic', 'who', 'have', 'free', 'time', 'to', 'wait', 'in', 'line', 'on', 'sunday', 'night', 'are', 'not', 'a', 'repres', 'sampl']

Raw tweet: 1-6 TIX Calgary Flames vs COL Avalanche Preseason 9/29 Scotiabank Saddledome http://t.co/5G8qA6mPxm
Tokenized and stemmed: ['tix', 'calgari', 'flame', 'vs', 'col', 'avalanch', 'preseason', 'scotiabank', 'saddledom']

Train matrix shape: (7613, 11635)
Test matrix shape:  (3263, 11635)
First 20 stems in vocabulary: ['00pm' '05th' '066gp' '06jst' '0fsloth' '0npzp' '0sed' '1-1st' '1000s'
 '1008pla' '1008planet' '100bn' '10

Save

In [3]:

df_X_train_counts = pd.DataFrame(X_train_counts.todense(), columns=vectorizer.get_feature_names_out())
df_X_train_counts.to_csv("data/X_train_counts.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)